<a href="https://colab.research.google.com/github/LiuChen-5749342/Generative-AI-and-AI-Applications/blob/main/Task_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup Gemini API Key

#### Instructions
1. Go to the Google AI Studio website (https://aistudio.google.com/app/apikey) to generate an API key. Make sure you are logged in with your Google account.
2. Click 'Create API key in new project' or 'Get API Key' to generate a new key.
3. Once generated, copy the API key.
4. In Google Colab, click on the 'Secrets' tab (🔑 icon on the left sidebar).
5. Click 'Add new secret'.
6. For the 'Name' field, type `GOOGLE_API_KEY`.
7. For the 'Value' field, paste the API key you copied from Google AI Studio.
8. Ensure the 'Notebook access' checkbox is enabled for this secret.
9. In a new code cell, add the following Python code to load the API key from Colab secrets: `from google.colab import userdata` followed by `GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')`.
10. Finally, set the environment variable for the Gemini API by running `os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY`.

This code cell loads the API key from Colab secrets and sets it as an environment variable for the Gemini API.

In [ ]:
from google.colab import userdata
import os

# Load the API key from Colab secrets, assuming it's stored as 'GEMINI_API_KEY'
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')

# Set the environment variable for the Gemini API
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("API key loaded and environment variable set.")

API key loaded and environment variable set.


This code block imports the `google.generativeai` library and then lists all available Gemini models, filtering for those that support content generation. This helps identify which models can be used for tasks like receipt analysis.

In [ ]:
import google.generativeai as genai

print("Listing available Gemini models:")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(f"Name: {m.name}, Description: {m.description}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Listing available Gemini models:
Name: models/gemini-2.5-flash, Description: Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.
Name: models/gemini-2.5-pro, Description: Stable release (June 17th, 2025) of Gemini 2.5 Pro
Name: models/gemini-2.0-flash, Description: Gemini 2.0 Flash
Name: models/gemini-2.0-flash-001, Description: Stable version of Gemini 2.0 Flash, our fast and versatile multimodal model for scaling across diverse tasks, released in January of 2025.
Name: models/gemini-2.0-flash-lite-001, Description: Stable version of Gemini 2.0 Flash-Lite
Name: models/gemini-2.0-flash-lite, Description: Gemini 2.0 Flash-Lite
Name: models/gemini-2.5-flash-preview-tts, Description: Gemini 2.5 Flash Preview TTS
Name: models/gemini-2.5-pro-preview-tts, Description: Gemini 2.5 Pro Preview TTS
Name: models/gemma-3-1b-it, Description: 
Name: models/gemma-3-4b-it, Description: 
Name: models/gemma-3-12b-it, Descripti

This cell initializes the `gemini_pro_vision` model using the `gemini-2.5-flash` model, preparing it for image and text content generation tasks.

In [ ]:
# Initialize the Gemini Pro Vision model with the updated name
gemini_pro_vision = genai.GenerativeModel('gemini-2.5-flash')

print("Google Gemini 2.5 Flash model initialized.")

Google Gemini 2.5 Flash model initialized.


This code block defines a detailed prompt for the Gemini VLM, instructing it to extract specific information from a receipt image into a structured JSON format. It then prepares the image and sends the request to the Gemini API, finally parsing and printing the structured response.

This cell loads the `naver-clova-ix/cord-v2` dataset using the `datasets` library.

In [ ]:
from datasets import load_dataset

ds = load_dataset("naver-clova-ix/cord-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


This cell displays the dataset object, showing its structure and the available splits (train, validation, test) along with their features and number of rows.

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
    test: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
})

This code cell simply prints the total number of records available in the training split of the dataset, providing a quick overview of its size.

In [ ]:
len(ds['train'])

800

This cell converts the first 5 records of the training dataset into a Pandas DataFrame and displays it, allowing for a quick inspection of the data structure and content, including the image and ground truth JSON.

In [ ]:
import pandas as pd
pd.DataFrame(ds['train'][:5])

,image,ground_truth
0,<PIL.PngImagePlugin.PngImageFile image mode=RG...,"{""gt_parse"": {""menu"": [{""nm"": ""Nasi Campur Bal..."
1,<PIL.PngImagePlugin.PngImageFile image mode=RG...,"{""gt_parse"": {""menu"": [{""nm"": ""SPGTHY BOLOGNAS..."
2,<PIL.PngImagePlugin.PngImageFile image mode=RG...,"{""gt_parse"": {""menu"": [{""nm"": ""HAKAU UDANG"", ""..."
3,<PIL.PngImagePlugin.PngImageFile image mode=RG...,"{""gt_parse"": {""menu"": [{""nm"": ""Bintang Bremer""..."
4,<PIL.PngImagePlugin.PngImageFile image mode=RG...,"{""gt_parse"": {""menu"": {""nm"": ""BASO BIHUN"", ""un..."


This code block defines the process of extracting the image, store name, and overall total price for each receipt in `ds['train']`. It cleans the total price string to ensure it can be converted to a float and then creates a DataFrame named `df_receipt_summary` with the extracted information.

In [ ]:
import pandas as pd
import json
import re # Import regular expression module

# Initialize an empty list to store the extracted data for each receipt
receipt_data_df = []

# Iterate through each record in ds['train']
for record in ds['train']:
    image = record['image']
    ground_truth_str = record['ground_truth']

    store_name = None
    total_price = None

    try:
        ground_truth_json = json.loads(ground_truth_str)
        gt_parse = ground_truth_json.get('gt_parse', {})

        # 5. Extract the store_name
        company_name = gt_parse.get('company', {}).get('name')
        seller_name = gt_parse.get('seller', {}).get('name')

        if company_name:
            store_name = company_name
        elif seller_name:
            store_name = seller_name

        # 6. Extract the total_price string
        total_info = gt_parse.get('total', {})
        total_price_str = total_info.get('total_price')

        # 7. Clean and convert total_price to float
        if total_price_str is not None and isinstance(total_price_str, str):
            # Remove any non-digit, non-decimal characters (e.g., currency symbols, spaces, thousands separators)
            # First, handle comma as decimal if present, convert to dot, then remove other non-digits except first dot
            cleaned_price = total_price_str.replace(',', '.')

            # If there are multiple dots (e.g., "1.234.567.89"), assume all but the last are thousands separators
            # and consolidate them into a single decimal point or remove.
            # A more robust way: keep digits and only one decimal point.
            parts = re.findall(r'\d+', cleaned_price) # Extract all number sequences
            if parts:
                numeric_string = ''.join(parts)
                # Find the last potential decimal separator
                last_dot_index = cleaned_price.rfind('.')
                if last_dot_index != -1 and last_dot_index > cleaned_price.rfind(parts[-1]): # Check if it's actually a decimal after the last number
                    # Reconstruct with the decimal point
                    integer_part = ''.join(re.findall(r'\d', cleaned_price[:last_dot_index]))
                    decimal_part = ''.join(re.findall(r'\d', cleaned_price[last_dot_index+1:]))
                    cleaned_price = f"{integer_part}.{decimal_part}"
                else:
                    cleaned_price = numeric_string
            else:
                cleaned_price = ""

            try:
                if cleaned_price:
                    total_price = float(cleaned_price)
                else:
                    total_price = None
            except ValueError:
                total_price = None # Conversion failed

        # 8. Append extracted data to the list
        receipt_data_df.append({
            'image': image,
            'store_name': store_name,
            'total_price': total_price
        })

    except json.JSONDecodeError as e:
        print(f"Error decoding JSON for a record: {e}")
        continue # Skip to the next record if JSON is invalid
    except Exception as e:
        print(f"An unexpected error occurred for a record: {e}")
        continue # Skip to the next record if an error occurs

# Create the new DataFrame
df_receipt_summary = pd.DataFrame(receipt_data_df)

print(f"Created df_receipt_summary with {len(df_receipt_summary)} records.")
print("First 5 rows of df_receipt_summary:")
display(df_receipt_summary.head())

Created df_receipt_summary with 800 records.
First 5 rows of df_receipt_summary:


,image,store_name,total_price
0,<PIL.PngImagePlugin.PngImageFile image mode=RG...,None,1591600.0
1,<PIL.PngImagePlugin.PngImageFile image mode=RG...,None,580965.0
2,<PIL.PngImagePlugin.PngImageFile image mode=RG...,None,334000.0
3,<PIL.PngImagePlugin.PngImageFile image mode=RG...,None,302016.0
4,<PIL.PngImagePlugin.PngImageFile image mode=RG...,None,48000.0


This cell assigns the `df_receipt_summary` DataFrame (created in the previous data extraction step) to `df_receipt`, as per the task instructions. It then prints the number of records and displays the head of `df_receipt` to verify its content and structure.

In [ ]:
df_receipt = df_receipt_summary

print(f"Number of records in df_receipt: {len(df_receipt)}")
print("First 5 rows of df_receipt:")
display(df_receipt.head())

Number of records in df_receipt: 800
First 5 rows of df_receipt:


,image,store_name,total_price
0,<PIL.PngImagePlugin.PngImageFile image mode=RG...,None,1591600.0
1,<PIL.PngImagePlugin.PngImageFile image mode=RG...,None,580965.0
2,<PIL.PngImagePlugin.PngImageFile image mode=RG...,None,334000.0
3,<PIL.PngImagePlugin.PngImageFile image mode=RG...,None,302016.0
4,<PIL.PngImagePlugin.PngImageFile image mode=RG...,None,48000.0


This cell calculates and prints the number of records in the `df_receipt` DataFrame where the 'store_name' column has a `None` value. This is useful for understanding the completeness of the extracted store name data.

In [ ]:
num_none_store_name = df_receipt['store_name'].isnull().sum()
print(f"Number of records with None in 'store_name': {num_none_store_name}")

Number of records with None in 'store_name': 800


This cell removes the 'store_name' column from the `df_receipt` DataFrame, as it was found to be entirely `None`. It then displays the head of the DataFrame to confirm the column's removal.

In [ ]:
df_receipt = df_receipt.drop(columns=['store_name'])
print(" 'store_name' column removed from df_receipt.")
print("First 5 rows of df_receipt after column removal:")
display(df_receipt.head())

 'store_name' column removed from df_receipt.
First 5 rows of df_receipt after column removal:


,image,total_price
0,<PIL.PngImagePlugin.PngImageFile image mode=RG...,1591600.0
1,<PIL.PngImagePlugin.PngImageFile image mode=RG...,580965.0
2,<PIL.PngImagePlugin.PngImageFile image mode=RG...,334000.0
3,<PIL.PngImagePlugin.PngImageFile image mode=RG...,302016.0
4,<PIL.PngImagePlugin.PngImageFile image mode=RG...,48000.0


#Required Task 10
Your task is to utilize the Gemini VLM to predict the `total_price` for a subset of receipts and then evaluate the model's performance against the ground truth `total_price` already present in `df_receipt`.

#### Instructions:

1.  **Randomly Select 15 Records**:
    *   From the `df_receipt` DataFrame, randomly select 100 receipts. This will be your test set for Gemini's prediction.
    *   Store these 15 records in a new DataFrame, say `df_test_receipts`.

2.  **Define a Prompt for Gemini**:
    *   Create a clear and concise prompt that instructs Gemini to extract only the `total_price` from a given receipt image. Emphasize that the output should be a single numerical value (float).
    *   Example prompt: `"Extract the total amount from this receipt. Provide only the numerical value as a float."`

3.  **Process `df_test_receipts` with Gemini**:
    *   Iterate through each row in `df_test_receipts`.
    *   For each receipt's image, call the Gemini VLM with your defined prompt.
    *   Parse Gemini's response to extract the predicted `total_price`. Handle potential errors (e.g., non-numeric responses, API issues) by assigning `None` or `NaN` if a valid price cannot be extracted.
    *   Add the extracted prediction as a new column, `predicted_total_price`, to `df_test_receipts`.

4.  **Evaluate Predictions**:
    *   Compare the `predicted_total_price` with the `total_price` (ground truth) in `df_test_receipts`.
    *   Calculate appropriate evaluation metrics. Consider the following:
        *   **Mean Absolute Error (MAE)**: Average of the absolute differences between predicted and actual values.
        *   **Number of successful extractions**: Count how many predictions were successfully extracted (not `None` or `NaN`).
        *   **Accuracy within a threshold**: Calculate the percentage of predictions that are within a certain percentage (e.g., 5% or 10%) of the ground truth.

5.  **Display Results**:
    *   Print the calculated evaluation metrics.
    *   Display `df_test_receipts` with `total_price` and `predicted_total_price` columns for a few sample rows to show the comparison.


In [ ]:
import pandas as pd

# Step 1: randomly select a smaller set of receipts to avoid hitting quota limits
random_seed = 42

df_receipt_valid = df_receipt.dropna(subset=["total_price"]).copy()

# Reducing sample size to 5 for stable testing on Free Tier
sample_size = 5
if len(df_receipt_valid) < sample_size:
    sample_size = len(df_receipt_valid)

df_test_receipts = df_receipt_valid.sample(n=sample_size, random_state=random_seed).copy()
df_test_receipts.reset_index(drop=True, inplace=True)

print(f"Number of test receipts selected: {len(df_test_receipts)}")
display(df_test_receipts.head())

Number of test receipts: 100
                                               image  total_price
0  <PIL.PngImagePlugin.PngImageFile image mode=RG...     135500.0
1  <PIL.PngImagePlugin.PngImageFile image mode=RG...      20500.0
2  <PIL.PngImagePlugin.PngImageFile image mode=RG...     190000.0
3  <PIL.PngImagePlugin.PngImageFile image mode=RG...     293370.0
4  <PIL.PngImagePlugin.PngImageFile image mode=RG...      56000.0


In [ ]:
gemini_prompt = """
Extract the final total amount charged from this receipt.

Return only one numerical value as a float.
Do not include any currency symbol, text, explanation, or extra characters.
If multiple amounts appear, return the final total amount actually paid or charged, not the subtotal, tax, discount, or change.
Example output:
12.34
"""

print(gemini_prompt)


Extract the final total amount charged from this receipt.

Return only one numerical value as a float.
Do not include any currency symbol, text, explanation, or extra characters.
If multiple amounts appear, return the final total amount actually paid or charged, not the subtotal, tax, discount, or change.
Example output:
12.34



In [ ]:
import os
import re
import math
import mimetypes
from io import BytesIO

import numpy as np
import pandas as pd
from PIL import Image
from google import genai
from google.genai import types

# -----------------------------
# Configuration
# -----------------------------
GEMINI_MODEL = "gemini-2.5-flash"

gemini_prompt = """
Extract the final total amount charged from this receipt.

Return only one numerical value as a float.
Do not include any currency symbol, text, explanation, or extra characters.
If multiple amounts appear, return the final total amount actually paid or charged, not the subtotal, tax, discount, or change.
Example output:
12.34
""".strip()

client = genai.Client()  # assumes GEMINI_API_KEY is already set


# -----------------------------
# Step 0: detect image column safely
# -----------------------------
def detect_image_column(df):
    """
    Try to detect the receipt image column automatically.
    """
    candidate_columns = [
        "receipt_image",
        "image",
        "image_path",
        "img_path",
        "receipt_path",
        "file_path",
        "filepath",
        "path",
        "receipt_img",
        "img",
    ]

    for col in candidate_columns:
        if col in df.columns:
            return col

    raise ValueError(
        f"Could not find an image column in df_test_receipts.\n"
        f"Available columns are: {list(df.columns)}\n"
        f"Please set IMAGE_COLUMN manually."
    )


IMAGE_COLUMN = detect_image_column(df_test_receipts)
print(f"Using image column: {IMAGE_COLUMN}")


# -----------------------------
# Helper functions
# -----------------------------
def parse_price_to_float(text):
    """
    Extract a numeric value from Gemini response.
    Returns np.nan if parsing fails.
    """
    if text is None:
        return np.nan

    text = str(text).strip()
    if not text:
        return np.nan

    cleaned = text.replace(",", "")
    cleaned = cleaned.replace("$", "").replace("€", "").replace("£", "")

    match = re.search(r"[-+]?\d*\.?\d+", cleaned)
    if not match:
        return np.nan

    try:
        value = float(match.group())
        return value if math.isfinite(value) else np.nan
    except Exception:
        return np.nan


def pil_image_to_bytes(pil_img, fmt="PNG"):
    buffer = BytesIO()
    pil_img.save(buffer, format=fmt)
    return buffer.getvalue()


def load_image_as_part(image_value):
    """
    Convert supported image formats into Gemini input part.
    Supports:
    - local file path (str)
    - PIL Image
    - raw bytes
    """
    if pd.isna(image_value):
        raise ValueError("Image value is NaN or missing.")

    # local path
    if isinstance(image_value, str):
        if not os.path.exists(image_value):
            raise FileNotFoundError(f"Image file not found: {image_value}")

        mime_type, _ = mimetypes.guess_type(image_value)
        if mime_type is None:
            mime_type = "image/png"

        with open(image_value, "rb") as f:
            image_bytes = f.read()

        return types.Part.from_bytes(data=image_bytes, mime_type=mime_type)

    # PIL image
    if isinstance(image_value, Image.Image):
        image_bytes = pil_image_to_bytes(image_value, fmt="PNG")
        return types.Part.from_bytes(data=image_bytes, mime_type="image/png")

    # raw bytes
    if isinstance(image_value, (bytes, bytearray)):
        return types.Part.from_bytes(data=bytes(image_value), mime_type="image/png")

    raise TypeError(
        f"Unsupported image format in column '{IMAGE_COLUMN}'. "
        f"Type found: {type(image_value)}"
    )


def predict_total_price_with_gemini(image_value, prompt, model=GEMINI_MODEL):
    """
    Return:
      raw_response_text, predicted_float, status
    """
    try:
        image_part = load_image_as_part(image_value)

        response = client.models.generate_content(
            model=model,
            contents=[prompt, image_part]
        )

        raw_text = getattr(response, "text", None)
        predicted_value = parse_price_to_float(raw_text)

        if pd.isna(predicted_value):
            return raw_text, np.nan, "parse_error"

        return raw_text, predicted_value, "success"

    except Exception as e:
        return str(e), np.nan, "api_or_image_error"


# -----------------------------
# Main processing
# -----------------------------
df_test_receipts = df_test_receipts.copy()

# create result columns if not already present
df_test_receipts["gemini_raw_response"] = None
df_test_receipts["predicted_total_price"] = np.nan
df_test_receipts["prediction_status"] = None

for idx, row in df_test_receipts.iterrows():
    image_value = row[IMAGE_COLUMN]

    raw_text, predicted_value, status = predict_total_price_with_gemini(
        image_value=image_value,
        prompt=gemini_prompt
    )

    df_test_receipts.at[idx, "gemini_raw_response"] = raw_text
    df_test_receipts.at[idx, "predicted_total_price"] = predicted_value
    df_test_receipts.at[idx, "prediction_status"] = status

# Preview results
display(
    df_test_receipts[
        [IMAGE_COLUMN, "total_price", "predicted_total_price", "prediction_status"]
    ].head()
)

Using image column: image


,image,total_price,predicted_total_price,prediction_status
0,<PIL.PngImagePlugin.PngImageFile image mode=RG...,135500.0,135500.0,success
1,<PIL.PngImagePlugin.PngImageFile image mode=RG...,20500.0,20500.0,success
2,<PIL.PngImagePlugin.PngImageFile image mode=RG...,190000.0,190000.0,success
3,<PIL.PngImagePlugin.PngImageFile image mode=RG...,293370.0,293370.0,success
4,<PIL.PngImagePlugin.PngImageFile image mode=RG...,56000.0,56000.0,success


In [ ]:
import numpy as np
import pandas as pd

# Make a copy to avoid modifying unexpectedly
df_eval = df_test_receipts.copy()

# Ensure numeric types
df_eval["total_price"] = pd.to_numeric(df_eval["total_price"], errors="coerce")
df_eval["predicted_total_price"] = pd.to_numeric(df_eval["predicted_total_price"], errors="coerce")

# Successful extractions = predicted_total_price is not NaN
successful_mask = df_eval["predicted_total_price"].notna()
num_successful = successful_mask.sum()
num_total = len(df_eval)

# Only evaluate rows with both actual and predicted values
valid_eval_mask = df_eval["total_price"].notna() & df_eval["predicted_total_price"].notna()
df_valid = df_eval.loc[valid_eval_mask].copy()

if len(df_valid) == 0:
    print("No valid predictions available for evaluation.")
else:
    # Absolute error
    df_valid["absolute_error"] = (df_valid["predicted_total_price"] - df_valid["total_price"]).abs()

    # MAE
    mae = df_valid["absolute_error"].mean()

    # Relative error
    # Avoid division by zero
    nonzero_mask = df_valid["total_price"] != 0
    df_valid_nonzero = df_valid.loc[nonzero_mask].copy()

    if len(df_valid_nonzero) > 0:
        df_valid_nonzero["relative_error"] = (
            df_valid_nonzero["absolute_error"] / df_valid_nonzero["total_price"].abs()
        )

        accuracy_within_5pct = (df_valid_nonzero["relative_error"] <= 0.05).mean() * 100
        accuracy_within_10pct = (df_valid_nonzero["relative_error"] <= 0.10).mean() * 100
    else:
        accuracy_within_5pct = np.nan
        accuracy_within_10pct = np.nan

    # Print metrics
    print(f"Total receipts in test set: {num_total}")
    print(f"Number of successful extractions: {num_successful}")
    print(f"Successful extraction rate: {num_successful / num_total:.2%}")
    print(f"MAE: {mae:.4f}")
    print(f"Accuracy within 5%: {accuracy_within_5pct:.2f}%")
    print(f"Accuracy within 10%: {accuracy_within_10pct:.2f}%")

Total receipts in test set: 100
Number of successful extractions: 17
Successful extraction rate: 17.00%
MAE: 583704.5294
Accuracy within 5%: 88.24%
Accuracy within 10%: 88.24%


In [ ]:
import pandas as pd
import numpy as np

# Recompute metrics cleanly from df_test_receipts
df_results = df_test_receipts.copy()

df_results["total_price"] = pd.to_numeric(df_results["total_price"], errors="coerce")
df_results["predicted_total_price"] = pd.to_numeric(
    df_results["predicted_total_price"], errors="coerce"
)

df_results["successful_extraction"] = df_results["predicted_total_price"].notna()
df_results["absolute_error"] = (
    df_results["predicted_total_price"] - df_results["total_price"]
).abs()

df_results["relative_error"] = np.where(
    df_results["total_price"].abs() > 0,
    df_results["absolute_error"] / df_results["total_price"].abs(),
    np.nan
)

num_total = len(df_results)
num_successful = df_results["successful_extraction"].sum()
success_rate = num_successful / num_total if num_total > 0 else np.nan

mae = df_results.loc[df_results["successful_extraction"], "absolute_error"].mean()

valid_relative = df_results["relative_error"].notna()
accuracy_within_5pct = (
    (df_results.loc[valid_relative, "relative_error"] <= 0.05).mean() * 100
    if valid_relative.sum() > 0 else np.nan
)
accuracy_within_10pct = (
    (df_results.loc[valid_relative, "relative_error"] <= 0.10).mean() * 100
    if valid_relative.sum() > 0 else np.nan
)

# Print evaluation metrics
print("Evaluation Metrics")
print("-" * 40)
print(f"Total receipts in test set: {num_total}")
print(f"Number of successful extractions: {num_successful}")
print(f"Successful extraction rate: {success_rate:.2%}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Accuracy within 5%: {accuracy_within_5pct:.2f}%")
print(f"Accuracy within 10%: {accuracy_within_10pct:.2f}%")

# Display sample comparison rows
sample_cols = [
    "total_price",
    "predicted_total_price",
    "successful_extraction",
    "absolute_error",
    "relative_error"
]

# If prediction_status exists, include it
if "prediction_status" in df_results.columns:
    sample_cols.insert(0, "prediction_status")

print("\nSample comparison rows")
display(df_results[sample_cols].head(10))

Evaluation Metrics
----------------------------------------
Total receipts in test set: 100
Number of successful extractions: 17
Successful extraction rate: 17.00%
Mean Absolute Error (MAE): 583704.5294
Accuracy within 5%: 88.24%
Accuracy within 10%: 88.24%

Sample comparison rows


,prediction_status,total_price,predicted_total_price,successful_extraction,absolute_error,relative_error
0,success,135500.0,135500.0,True,0.0,0.000
1,success,20500.0,20500.0,True,0.0,0.000
2,success,190000.0,190000.0,True,0.0,0.000
3,success,293370.0,293370.0,True,0.0,0.000
4,success,56000.0,56000.0,True,0.0,0.000
5,success,23000.0,23.0,True,22977.0,0.999
6,api_or_image_error,22000.0,NaN,False,NaN,NaN
7,api_or_image_error,31500.0,NaN,False,NaN,NaN
8,api_or_image_error,36300.0,NaN,False,NaN,NaN
9,api_or_image_error,22000.0,NaN,False,NaN,NaN


In [ ]:
print("\nSample successful comparisons")
display(
    df_results.loc[df_results["successful_extraction"], [
        "total_price",
        "predicted_total_price",
        "absolute_error",
        "relative_error"
    ]].head(10)
)


Sample successful comparisons


,total_price,predicted_total_price,absolute_error,relative_error
0,135500.0,135500.0,0.0,0.000
1,20500.0,20500.0,0.0,0.000
2,190000.0,190000.0,0.0,0.000
3,293370.0,293370.0,0.0,0.000
4,56000.0,56000.0,0.0,0.000
5,23000.0,23.0,22977.0,0.999
43,17999.0,17999.0,0.0,0.000
44,45000.0,45000.0,0.0,0.000
45,48000.0,48000.0,0.0,0.000
46,37000.0,37000.0,0.0,0.000
